Due to the current situation (`Updated 09/02/2026`),

- Google Gemini has reduced the rate limits for several models, such as `gemini-2.5-flash` and `gemini-3-flash` (text models used in Colab notebooks), to a **limit of 20 Requests Per Day (RPD)**.

- To continue using these models seamlessly with sufficient rate limits, it is necessary to upgrade to the **pay-as-you-go tier** (link a Billing Account).
  - 👉 You can learn how to do this here: [https://ai.google.dev/gemini-api/docs/billing](https://ai.google.dev/gemini-api/docs/billing)

- Alternatively, you can follow the Groq API approach described below.

- Update `23/08/2026`: Removed the `llama` models because they are no longer supported by Groq, and replaced them with `qwen/qwen3.6-27b`.


Most of concepts and codes are adapted from this [repo](https://github.com/dair-ai/Prompt-Engineering-Guide).

Implementation Detail:
- LangChain is used.

# Setting environments and model setup

In [1]:
from IPython.display import display, Markdown

## Approach 1: Gemini

In [ ]:
# %%capture
# !pip install -qU langchain-google-genai

Request for Google API KEY here : https://aistudio.google.com/app/apikey

In [ ]:
# from getpass import getpass
# import os

# if "GOOGLE_API_KEY" not in os.environ:
#     os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI API key: ")

Since this is an open-ended generation, we do not want the generation to be boring so temperature is set to 1 instead of 0.

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=1,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

## Approach 2: Groq API


However, we still have an **alternative** that can be used via a free-tier API: **Groq API** (compatible with LangChain). This does not require linking a credit card and offers several models, such as:

Available models: https://console.groq.com/settings/limits

👉 You can sign up and get your API Key here: [https://console.groq.com/keys](https://console.groq.com/keys)


In [2]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.3 MB/s eta 0:00:00


In [3]:
import getpass
import os

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


Since this is an open-ended generation, we do not want the generation to be boring so temperature is set to 1 instead of 0.

In [27]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="none",
    max_tokens=600,
    timeout=None,
    max_retries=2,
)

# Zero-shot / Few-shot Prompting

**Ex. Summarize the customer feedback.**

In [28]:
# Zero-shot

prompt = """Summarize the customer feedback.
Complaint:
The room was comfortable and had a good view.
However, the air conditioner was very noisy at night.
I contacted reception, but no technician arrived.
This made it difficult for me to sleep.
The hotel should respond to maintenance requests more quickly.
Summary:"""

zero_shot = llm.invoke(prompt)
display(Markdown(zero_shot.content))

While the guest appreciated the comfortable room and good view, they were dissatisfied due to a noisy air conditioner that disrupted their sleep. The issue was exacerbated by the lack of response from the reception and the failure of a technician to arrive, leading to criticism regarding the hotel's slow maintenance response times.

The output is in a free-form format. If you want the model to follow a specific structure, you can:
- Add output-format constraints directly to your prompt.
or
- Provide a few-shot example showing the desired format. (We will try this next).

In [29]:
# Few-shot

prompt = """Summarize the feedback following the example.

–Example 1–

Complaint:
The hotel room was clean, and the staff were friendly.
However, check-in took almost 40 minutes.
The room was also not ready at the promised time.
I had to wait in the lobby with my luggage.
The hotel should improve its check-in process.

Summary:
Positive: Clean room and friendly staff.

Issue: Long check-in time and delayed room availability.

Suggestion: Improve the check-in and room preparation process.

–Example 2–

Complaint:
The breakfast had a good variety of food.
However, several dishes were already cold.
The staff took a long time to refill empty items.
There were also not enough tables during busy hours.
The hotel should improve its breakfast service.

Summary:
Positive: Good variety of breakfast options.

Issue: Cold food, slow refills, and insufficient seating.

Suggestion: Improve food temperature, refill speed, and seating availability.

–Now–
Complaint:
The room was comfortable and had a good view.
However, the air conditioner was very noisy at night.
I contacted reception, but no technician arrived.
This made it difficult for me to sleep.
The hotel should respond to maintenance requests more quickly.
Summary:
"""

In [30]:
few_shot = llm.invoke(prompt)
display(Markdown(few_shot.content))

Positive: Comfortable room and good view.

Issue: Noisy air conditioner disrupting sleep and lack of technician response.

Suggestion: Improve the speed of maintenance request responses.

# Chain of Thought Prompting

**Example: Marketing Strategy Planning**

In [31]:
input_text = """
Should we spend the entire 500,000 THB marketing budget on Facebook Ads,
or should we distribute it across TikTok and Google Search as well?
Come up with an analysis and diversified plan suggestion.
"""

**1.) Without chain-of-thought**

In [32]:
prompt = f"""
{input_text}

Return the output within 200 words.
Explain in simple language.
"""

ai_msg_no_reasoning = llm.invoke(prompt)

display(Markdown("## Without Chain-of-Thought"))
display(Markdown(ai_msg_no_reasoning.content))

## Without Chain-of-Thought

Spending the entire 500,000 THB on Facebook Ads is risky. Relying on one platform limits your reach and makes you vulnerable to algorithm changes or rising costs. A diversified strategy is smarter because it targets customers at different stages of their buying journey.

Here is a suggested plan:

1.  **Facebook/Instagram (40% - 200,000 THB):** Use this for broad brand awareness and retargeting. Facebook remains strong for establishing trust and reaching a wide, local audience.
2.  **TikTok (30% - 150,000 THB):** Allocate this to capture younger demographics and drive viral engagement. TikTok is excellent for top-of-funnel discovery, creating buzz, and showcasing products in an entertaining way.
3.  **Google Search (30% - 150,000 THB):** Save this for high-intent users. Google Ads target people actively searching for what you sell. This channel typically has the highest conversion rate because it captures demand rather than just creating it.

By splitting the budget, you create a balanced funnel: TikTok attracts attention, Facebook builds familiarity, and Google converts ready-to-buy customers. This approach reduces risk, maximizes overall reach, and ensures you capture customers regardless of where they spend their time online. Test each channel for two weeks, then shift funds toward the best-performing platform while maintaining a presence on all three for stability.

**Next, Zero-shot Chain-of-Thought :** `Let's think step-by-step`

In [33]:
prompt = f"""
{input_text}
Let's think step-by-step.
Return the output within 200 words.
Explain in simple language.
"""

ai_msg_no_reasoning = llm.invoke(prompt)

display(Markdown("## Zero-shot Chain-of-Thought"))
display(Markdown(ai_msg_no_reasoning.content))

## Zero-shot Chain-of-Thought

Spending your entire 500,000 THB budget solely on Facebook Ads is risky. While Facebook excels at targeted advertising, relying on one platform limits your reach and exposes you to sudden algorithm changes or rising costs. A diversified approach is smarter because different platforms serve different stages of the customer journey.

Here is a suggested split:

1.  **Facebook (50% - 250,000 THB):** Use this for broad awareness and retargeting. Facebook’s robust targeting helps you find interested users and remind them to buy.
2.  **TikTok (30% - 150,000 THB):** Allocate this for viral reach and engaging younger audiences. TikTok is excellent for building brand personality and generating organic interest through creative video content.
3.  **Google Search (20% - 100,000 THB):** Reserve this for high-intent customers. When people search for your product, they are ready to buy. Google captures this immediate demand, ensuring high conversion rates.

By splitting the budget, you balance brand building with direct sales. Facebook keeps your brand top-of-mind, TikTok attracts new attention, and Google closes the sale from eager buyers. This strategy reduces risk, maximizes overall reach, and ensures you don’t miss customers at any stage of their decision-making process. Start with this split, monitor the performance of each channel for two weeks, and then adjust the amounts based on which platform delivers the best results.

> Note: **Nowadays, many models use internal or implicit chain-of-thought reasoning by default**, so simply adding a ```zero-shot chain-of-thought``` prompt **may not** make a significant difference.

**2.) With your chain-of-thought**

> Thinking about :
- Product → Target Customer → Marketing Funnel → Budget Allocation → Risks → Final Recommendation

The answer will be improved.

In [34]:
prompt_reasoning = f"""
{input_text}

Before answering, reason through the following steps:

1. Identify the product/service and target customers.
   - If this information is unknown, assume two clearly different cases
     and provide separate recommendations.

2. Consider which stage of the marketing funnel each channel is best suited for:
   - Awareness
   - Consideration
   - Conversion

3. Estimate how the 500,000 THB budget could be allocated across channels.
   Consider whether each allocation is sufficient for test-and-learn.

4. Analyze the risks of each strategy.

5. Provide a final recommendation with an approximate budget allocation.

Return the output within 200 words.
Explain in simple language.
"""

ai_msg_reasoning = llm.invoke(prompt_reasoning)

display(Markdown("## With Chain-of-Thought"))
display(Markdown(ai_msg_reasoning.content))

## With Chain-of-Thought

Since the product is unknown, let’s assume a **B2C e-commerce brand** targeting young adults.

**Funnel Fit:**
*   **TikTok:** Best for *Awareness*. It generates viral reach and creates hype.
*   **Facebook:** Best for *Consideration*. It allows precise targeting to retarget interested users.
*   **Google Search:** Best for *Conversion*. It captures high-intent users actively searching for your product.

**Budget Allocation Strategy:**
Spending 500,000 THB solely on Facebook is risky. You miss out on TikTok’s viral potential and Google’s high-intent traffic. A diversified approach ensures you cover the entire customer journey.

**Suggested Split (Test-and-Learn Phase):**
*   **30% (150,000 THB) on TikTok:** Create engaging short videos to build brand awareness.
*   **40% (200,000 THB) on Facebook:** Use lookalike audiences and retargeting to nurture leads.
*   **30% (150,000 THB) on Google Search:** Bid on specific keywords to capture ready-to-buy customers.

**Risks:**
*   **All-in on Facebook:** You ignore emerging trends and high-intent searches, potentially lowering overall ROI.
*   **Over-diversification:** If budgets are too small per channel, data may be insufficient to optimize ads. However, 150k+ THB per channel is enough for meaningful testing.

**Final Recommendation:**
Do not put all eggs in one basket. Use the **30-40-30 split**. This balances brand building (TikTok), audience refinement (Facebook), and direct sales (Google). Monitor performance weekly; shift budget toward the channel delivering the lowest Cost Per Acquisition (CPA) after the first month. This diversified strategy minimizes risk while maximizing reach across different customer intent stages.

You can see this **difference** in the response:

`Since your product isn’t specified, let’s look at two common scenarios.`

- **Before CoT**: The **model mainly thinks about how to diversify** the budget.
- **After CoT**: The** model first considers the product **and target customer, **makes assumptions when needed**, then **analyzes the options** and gives a recommendation.

This **improves the depth and structure of the analysis**.

**3.) Few-shot Chain-of-Thought (Geometry Problem)**
> In cases where we want the reasoning steps to follow a clear structure — instead of letting the model generate its own free-form chain-of-thought — we can provide a few labeled examples to guide the reasoning process.
This technique is called few-shot chain-of-thought (few-shot CoT).

In [35]:
few_shot_cot_prompt = """Solve the following math problems step by step:

Example 1:
Problem: If a train travels at 60 miles per hour for 2.5 hours, how far does it go?
Solution:
1. Identify the given information:
   - Speed of the train: 60 miles per hour
   - Time of travel: 2.5 hours
2. Use the formula: Distance = Speed × Time
3. Plug in the values:
   Distance = 60 miles/hour × 2.5 hours
4. Calculate:
   Distance = 150 miles
Therefore, the train travels 150 miles.

Problem: A bakery sold 136 cakes last week. This week, they sold 25% more. How many cakes did they sell this week?
Solution:
1. Identify the given information:
   - Last week's sales: 136 cakes
   - Increase: 25%
2. Calculate the increase:
   25% of 136 = 0.25 × 136 = 34 cakes
3. Add the increase to last week's sales:
   This week's sales = 136 + 34 = 170 cakes
Therefore, the bakery sold 170 cakes this week.

Here is a new question:
Problem: If a rectangle has a length of 15 meters and a width of 8 meters, what is its area?
Solution:
"""

In [36]:
ans = llm.invoke(few_shot_cot_prompt).content
display(Markdown(ans))

Solution:
1. Identify the given information:
   - Length of the rectangle: 15 meters
   - Width of the rectangle: 8 meters
2. Use the formula: Area = Length × Width
3. Plug in the values:
   Area = 15 meters × 8 meters
4. Calculate:
   Area = 120 square meters
Therefore, the area of the rectangle is 120 square meters.

# Additional Topics:

## Analogy-based Prompting

Concept:
> **Let the model recall similar problems from other contexts or domains**, then **transfer useful patterns or lessons** to the current problem.

Expectation:
> **Quickly improve response quality—such as specificity, groundedness, or idea interestingness**—without requiring heavy prompt engineering, such as manually designing detailed reasoning steps or preparing few-shot examples.

**Ex. Employee Adoption of an Internal AI Tool**

In [37]:
problem = """
A company has introduced a new internal AI assistant for employees.

Most employees try the tool once or twice, but then stop using it.
The company has already provided basic training, but adoption remains low.

What should the company do to increase long-term usage?
"""

**Without analogy-based prompting**

In [38]:
prompt = f"""
Problem:
{problem}

Recommend a practical solution.
Keep the response concise.
"""

res = llm.invoke(prompt)

display(Markdown("## Without Analogy-Based Prompting"))
display(Markdown(res.content))

## Without Analogy-Based Prompting

Shift from broad training to **context-specific integration**. Identify 2–3 high-friction, repetitive tasks common across departments (e.g., drafting meeting summaries or formatting reports) and embed the AI directly into the existing workflow tools where those tasks occur. Provide "just-in-time" prompts that solve these specific pain points immediately, demonstrating tangible time savings rather than abstract capabilities.

**With analogy-based prompting**

In [39]:
prompt = f"""
Problem:
{problem}

Before solving the problem:

1. Recall 2 similar situations from other domains where people
   tried a new product or behavior but failed to continue using it.

2. Explain what helped improve long-term adoption in those situations.

3. Identify which lessons can be transferred to this workplace problem.

4. Apply those lessons to recommend a practical strategy
   for increasing long-term AI tool usage.

Keep the response concise.

Relevant analogies:

Final recommendation:
"""

ai_msg_with = llm.invoke(prompt)

display(Markdown("## With Analogy-Based Prompting"))
display(Markdown(ai_msg_with.content))

## With Analogy-Based Prompting

Relevant analogies:
1. **Fitness Apps/Trackers**: Users buy devices or download apps with high initial enthusiasm but abandon them after a few weeks.
2. **New Software Suites (e.g., Slack/Zoom initially)**: Employees resist switching from established workflows (like email or in-person chats) due to friction and lack of immediate perceived value.

Final recommendation:
1. **Habit-Loop Integration**: Stop treating the AI as a separate tool. Embed it directly into existing high-frequency workflows (e.g., inside the email client or code editor) so usage requires zero context switching.
2. **Specific, High-Value Use Cases**: Move beyond general training. Provide "cheat sheets" for specific, time-consuming tasks (e.g., "Draft a meeting summary" or "Debug this Python error") to demonstrate immediate ROI.
3. **Gamified Feedback & Social Proof**: Implement lightweight metrics showing time saved. Highlight peer success stories (e.g., "Marketing team reduced report drafting time by 40%") to create social pressure and normalize usage.

- **Without Analogy**: More generic and less interesting recommendations, based mainly on the immediate problem.
- **With Analogy**: Encourages the model to think through similar cases, which can lead to more interesting, specific, and grounded ideas.

Helpful Website for Prompting Techniques : https://www.promptingguide.ai/techniques

## LLM-as-Judge

**1. Prepare the ideas to evaluate.**

In [40]:
import pandas as pd
import json

# Ideas to evaluate
ideas = {
    "Idea 1": """
Introduce AI technology to automatically read and process
documents and forms across the organization.

The goal is to reduce manual data-entry workload,
improve overall operational speed, and significantly reduce costs.

The development team will immediately experiment with free
open-source tools so that the project can deliver results quickly.
""",

    "Idea 2": """
Develop an automated OCR system for processing invoice forms
for the Accounting Department.

The goal is to reduce manual data-entry time by 40%
and reduce the data error rate to below 2% within this quarter.

The project will run as a 3-week pilot using
1 developer and 1 accounting staff member
before full deployment.
"""
}

**2. Define the Evaluation Rubric**
> Using a detailed scoring rubric with clearly specified criteria is recommended to improve the consistency of the LLM judge.

In [41]:
rubric = """
### Clarity
- Score 1: Vague or confusing; no clear problem or target audience.
- Score 2: Broad idea, but lacks a defined problem and specific audience.
- Score 3: Understandable, but lacks important details or focus.
- Score 4: Clear problem and audience, with minor gaps in structure or focus.
- Score 5: Extremely clear, specific, and well-structured.

### Business Value
- Score 1: No clear business benefit.
- Score 2: Potential benefit, but weak connection to business goals.
- Score 3: Good potential impact, but lacks measurable KPIs.
- Score 4: Clear business alignment and measurable KPIs,
  but ROI or concrete impact is still incomplete.
- Score 5: Strong business alignment, measurable KPIs,
  and clear ROI or business impact.

### Feasibility
- Score 1: Unrealistic and lacks actionable steps.
- Score 2: Some steps are proposed, but major resource or execution issues remain.
- Score 3: Actionable, but missing important details such as who, when, or how.
- Score 4: Clear execution plan with minor gaps in resources or timeline.
- Score 5: Highly actionable with clear steps,
  realistic resources, ownership, and timeline.
"""

**3. Run the LLM-as-Judge Loop**

In [42]:
results = []

for idea_name, idea in ideas.items():

    prompt = f"""
You are a strict but fair Business Strategy Director.

Evaluate the following business idea using the rubric below.

{rubric}

Idea:
{idea}

Return ONLY valid JSON in this format:

{{
  "clarity": 1,
  "clarity_rationale": "...",
  "business_value": 1,
  "business_value_rationale": "...",
  "feasibility": 1,
  "feasibility_rationale": "...",
  "final_suggestion": "..."
}}
"""

    response = llm.invoke(prompt).content.strip()

    # Remove Markdown code fences if the model adds them
    response = response.replace("```json", "").replace("```", "").strip()

    result = json.loads(response)

    results.append({
        "Idea": idea_name,
        "Clarity": result.get("clarity"),
        "Business Value": result.get("business_value"),
        "Feasibility": result.get("feasibility"),
        "Clarity Rationale": result.get("clarity_rationale", ""),
        "Business Value Rationale": result.get("business_value_rationale", ""),
        "Feasibility Rationale": result.get("feasibility_rationale", ""),
        "Final Suggestion": result.get("final_suggestion", "N/A")
    })

df = pd.DataFrame(results)
display(df)

,Idea,Clarity,Business Value,Feasibility,Clarity Rationale,Business Value Rationale,Feasibility Rationale,Final Suggestion
0,Idea 1,2,3,2,The idea is broad and lacks a defined problem ...,"Good potential impact, but lacks measurable KP...","Some steps are proposed, but major resource or...","Narrow the scope to a specific high-volume, lo..."
1,Idea 2,4,4,3,The problem (manual data entry inefficiency) a...,The idea aligns well with operational efficien...,"The proposal lists resources (1 dev, 1 staff) ...",Refine the feasibility plan by specifying the ...


**Designing LLM-as-Judge is iterative refinement work, you need to inspect the prompt <-> response of the judgement , calibrate the rubric to better align your task**



## Iterative Refinement

> Reflection Loop: We can integrate LLM-as-a-Judge into a refinement loop to automatically improve response quality based on the judge’s feedback.

Idea → Judge → Feedback → Refine → Re-evaluate → Compare Before vs. After

In [43]:
# Original low-scoring idea
idea_1 = """
Introduce AI technology to automatically read and process
documents and forms across the organization.

The goal is to reduce manual data-entry workload,
improve overall operational speed, and significantly reduce costs.

The development team will immediately experiment with free
open-source tools so that the project can deliver results quickly.
"""

**Detailed Rubric 1 to 5**

In [44]:
rubric = """
### Clarity
- Score 1: Vague or confusing; no clear problem or target audience.
- Score 2: Broad idea, but lacks a defined problem and specific audience.
- Score 3: Understandable, but lacks important details or focus.
- Score 4: Clear problem and audience, with minor gaps in structure or focus.
- Score 5: Extremely clear, specific, and well-structured.

### Business Value
- Score 1: No clear business benefit.
- Score 2: Potential benefit, but weak connection to business goals.
- Score 3: Good potential impact, but lacks measurable KPIs.
- Score 4: Clear business alignment and measurable KPIs,
  but ROI or concrete impact is still incomplete.
- Score 5: Strong business alignment, measurable KPIs,
  and clear ROI or business impact.

### Feasibility
- Score 1: Unrealistic and lacks actionable steps.
- Score 2: Some steps are proposed, but major resource or execution issues remain.
- Score 3: Actionable, but missing important details such as who, when, or how.
- Score 4: Clear execution plan with minor gaps in resources or timeline.
- Score 5: Highly actionable with clear steps,
  realistic resources, ownership, and timeline.
"""

**Judge Prompt Function**

In [45]:
def judge_idea(idea):

    prompt = f"""
You are a strict but fair Business Strategy Director.

Evaluate the following business idea using the rubric below.

{rubric}

Idea:
{idea}

Return ONLY valid JSON:

{{
  "clarity": 1,
  "business_value": 1,
  "feasibility": 1,
  "feedback": "Give concise and actionable feedback for improving the idea."
}}
"""

    response = llm.invoke(prompt).content.strip()

    response = (
        response
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    return json.loads(response)

**Run the Reflection Loop**

In [47]:
from tqdm.notebook import tqdm

steps = [
    "Evaluate original idea",
    "Reflect and improve",
    "Evaluate improved idea"
]

with tqdm(total=len(steps), desc="Reflection Loop") as pbar:

    # Step 1: Evaluate the original idea
    before = judge_idea(idea_1)
    pbar.update(1)

    # Step 2: Improve the idea using judge feedback
    refinement_prompt = f"""
Original Idea:
{idea_1}

Evaluator Feedback:
{before["feedback"]}

Revise the idea to address the feedback.

Improve:
- clarity and scope
- measurable business value
- feasibility and execution details

Do not change the core objective of using AI for document processing.

Return only the improved idea.
"""

    improved_idea = llm.invoke(refinement_prompt).content.strip()
    pbar.update(1)

    # Step 3: Evaluate the improved idea
    after = judge_idea(improved_idea)
    pbar.update(1)

Reflection Loop:   0%|          | 0/3 [00:00<?, ?it/s]

**Shows the idea : before vs. after**

In [48]:
comparison = pd.DataFrame([
    {
        "Version": "Before",
        "Clarity": before["clarity"],
        "Business Value": before["business_value"],
        "Feasibility": before["feasibility"],
    },
    {
        "Version": "After",
        "Clarity": after["clarity"],
        "Business Value": after["business_value"],
        "Feasibility": after["feasibility"],
    }
])

comparison["Average"] = comparison[
    ["Clarity", "Business Value", "Feasibility"]
].mean(axis=1)

display(comparison)

,Version,Clarity,Business Value,Feasibility,Average
0,Before,3,3,2,2.666667
1,After,5,5,4,4.666667


In [49]:
# Show the ideas and judge feedback

display(Markdown("## Before Refinement"))
display(Markdown(idea_1))

display(Markdown("### Judge Feedback"))
display(Markdown(before["feedback"]))

## Before Refinement


Introduce AI technology to automatically read and process
documents and forms across the organization.

The goal is to reduce manual data-entry workload,
improve overall operational speed, and significantly reduce costs.

The development team will immediately experiment with free
open-source tools so that the project can deliver results quickly.


### Judge Feedback

1. Define Scope: Specify which document types and departments are targets to establish a clear pilot. 2. Quantify Value: Establish baseline metrics for current processing time and cost to measure ROI. 3. Address Risk: 'Immediately experimenting' with free tools poses significant data security and integration risks. Propose a phased POC with defined security protocols and a clear migration path to production-grade solutions before full rollout.

In [50]:
display(Markdown("## After Refinement"))
display(Markdown(improved_idea))

display(Markdown("### Judge Feedback"))
display(Markdown(after["feedback"]))

## After Refinement

**Project Proposal: Phased AI-Powered Document Intelligence Pilot**

**Objective**
Implement an AI-driven document processing system to automate data extraction from high-volume forms, thereby reducing manual entry workload, accelerating operational throughput, and lowering processing costs.

**Scope & Pilot Definition**
*   **Target Department:** Finance and Accounts Payable.
*   **Document Types:** Standardized vendor invoices and employee expense reports.
*   **Pilot Duration:** 8 weeks.

**Measurable Business Value & Baselines**
To ensure clear ROI, the project will establish and track the following baseline metrics against post-implementation results:
*   **Processing Time:** Reduce average document processing time from 15 minutes (current manual baseline) to under 2 minutes per document.
*   **Accuracy:** Achieve >95% data extraction accuracy compared to human entry.
*   **Cost Reduction:** Target a 40% reduction in labor hours dedicated to manual data entry within the pilot department.

**Feasibility, Execution, & Risk Mitigation**
*   **Phased Approach:**
    1.  **Phase 1 (Weeks 1-2):** Conduct a security-compliant Proof of Concept (POC) using sandboxed, open-source tools on anonymized/dummy data only. No production data will be exposed.
    2.  **Phase 2 (Weeks 3-6):** Evaluate POC results against security protocols (data encryption, access controls) and integration capabilities. Select a production-grade solution based on these criteria.
    3.  **Phase 3 (Weeks 7-8):** Limited live rollout with dual-entry verification to validate accuracy before full organizational deployment.
*   **Security Protocol:** All data handling will adhere to existing enterprise data governance policies. A clear migration path from the experimental phase to a secure, scalable production environment is defined upfront to prevent "technical debt" or security vulnerabilities associated with ad-hoc tool usage.

### Judge Feedback

The proposal is exceptionally clear and defines strong KPIs. However, the feasibility score is capped at 4 because the 8-week timeline is aggressive for enterprise-grade integration, particularly regarding Vendor Management (procurement/legal review) and IT security sign-off. To reach a 5, specify the resource allocation (e.g., dedicated PM, DevOps support) and include a contingency plan for the 'dual-entry' validation phase, which often bottlenecks at scale. Ensure the 'open-source' POC strategy includes a clear exit criteria to avoid scope creep into custom development.

Now, the LLM can automatically optimize the idea against the given rubric through a closed-loop refinement process.